In [9]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [10]:
data_path = '../datasets/interim/filled_cleaned_train.csv'
df = pd.read_csv(data_path)
feature_cols = [
    'latitude', 'longitude', 'property_type', 'room_type', 'accommodates',
    'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value', 'instant_bookable'
]
for col in ['bathrooms', 'bedrooms', 'beds']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
available_features = [c for c in feature_cols if c in df.columns]
print(f"Features used: {len(available_features)} out of {len(feature_cols)} requested.")
X = df[available_features].copy()
y = df['price'].astype(float)
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print(f"Samples: {len(X)} | Features: {X.shape[1]}")
print(f"Numeric: {len(numeric_cols)} | Categorical: {len(categorical_cols)}")


Features used: 18 out of 18 requested.
Samples: 20804 | Features: 18
Numeric: 15 | Categorical: 3


In [11]:
from sklearn.metrics import mean_squared_error
import numpy as np
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])
model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)
valid_preds = model.predict(X_valid)
mae = mean_absolute_error(y_valid, valid_preds)
mse = mean_squared_error(y_valid, valid_preds)
rmse = np.sqrt(mse)
print(f"MAE: {mae:,.0f}")
print(f"RMSE: {rmse:,.0f}")


MAE: 2,922
RMSE: 17,120


In [ ]:
model.fit(X, y)
test_df = pd.read_csv('../datasets/test.csv')
test_ids = test_df['id']
for col in ['bathrooms', 'bedrooms', 'beds']:
    if col in test_df.columns:
        test_df[col] = pd.to_numeric(test_df[col], errors='coerce')
X_test = test_df[available_features].copy()
test_preds = model.predict(X_test)
submission = pd.DataFrame({'ID': test_ids, 'TARGET': test_preds})
submission_path = '../datasets/interim/submission_baseline.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {submission_path}')
submission.head()


Saved ../datasets/interim/submission.csv


,ID,TARGET
0,536526,2109.075000
1,124137,5210.136667
2,164216,3431.210000
3,541629,3077.706667
4,572504,2369.216667
